In [2]:
%%file producer.py
# Zadanie 2.1

from kafka import KafkaProducer
import json, random, time
from datetime import datetime
 
producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)
 
sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']
 
def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }
 
for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(0.5)
 
producer.flush()
producer.close()

Overwriting producer.py


In [3]:
%%file consumer_filter.py
# Zadanie 3.1

from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='risk-checker-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Nasłuchuję na duże transakcje (amount > 3000)...")

for message in consumer:
    tx = message.value
    if tx.get('amount') > 3000:
        tx_id = tx.get('tx_id')
        amount = tx.get('amount')
        store = tx.get('store')
        category = tx.get('category')
        
        print(f"ALERT: {tx_id} | {amount:.2f} PLN | {store} | {category}")

Overwriting consumer_filter.py


In [4]:
%%file consumer_enrich.py
# Zadanie 3.2

from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Rozpoczynam analizę poziomu ryzyka...")

for message in consumer:
    tx = message.value
    amount = tx.get('amount')
    
    if amount > 3000:
        risk_level = "HIGH"
    elif amount > 1000:
        risk_level = "MEDIUM"
    else:
        risk_level = "LOW"

    tx['risk_level'] = risk_level

    print(f"  {tx.get('tx_id')} | {amount} PLN |  {tx.get('store')} | {tx.get('risk_level')}")

Writing consumer_enrich.py


In [5]:
%%file consumer_count.py
# Zadanie 4.1

from kafka import KafkaConsumer
from collections import Counter
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = {}
msg_count = 0

for message in consumer:
    tx = message.value
    store = tx.get('store')
    amount = tx.get('amount')

    store_counts[store] += 1
    total_amount[store] = total_amount.get(store, 0.0) + amount
    msg_count += 1

    if msg_count % 10 == 0:
        print(f"\n--- Podsumowanie po {msg_count} wiadomościach ---")
        print(f"{'Sklep':<9} | {'Liczba':<4} | {'Suma [PLN]':<12} | {'Średnia':<10}")
        print("-" * 55)
        
        for s in sorted(store_counts.keys()):
            count = store_counts[s]
            total = total_amount[s]
            avg = total / count
            print(f"{s:<9} | {count:<4} | {total:<12.2f} | {avg:<10.2f}")

Overwriting consumer_count.py


In [6]:
%%file consumer_stats.py
# Zadanie 4.2

from kafka import KafkaConsumer
from collections import defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

category_stats = defaultdict(lambda: {
    'count': 0, 
    'total': 0.0, 
    'min': float('inf'), 
    'max': float('-inf')
})

msg_count = 0


for message in consumer:
    tx = message.value
    category = tx.get('category')
    amount = tx.get('amount')

    s = category_stats[category]
    s['count'] += 1
    s['total'] += amount
    
    if amount < s['min']:
        s['min'] = amount
    if amount > s['max']:
        s['max'] = amount

    msg_count += 1


    if msg_count % 10 == 0:
        print(f"\n--- Podsumowanie po {msg_count} wiadomościach ---")
        print(f"{'Kategoria':<12} | {'Liczba transakcji':<19} | {'Łączny przychód':<15} | {'Min':<8} | {'Max':<8}")
        print("-" * 55)
        
        for cat in sorted(category_stats.keys()):
            stats = category_stats[cat]
            print(f"{cat:<12} | {stats['count']:<19} | {stats['total']:<15.2f} | {stats['min']:<8.2f} | {stats['max']:<8.2f}")

Overwriting consumer_stats.py


In [7]:
%%file consumer_anomalies.py
# Praca domowa

from kafka import KafkaConsumer
from collections import defaultdict
import json
import time

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Nasłuchuję anomalii prędkości (więcej niż 3 transakcje / 60s)...")

user_history = defaultdict(list)

for message in consumer:
    tx = message.value
    user_id = tx.get('user_id')

    current_time = time.time()
    user_history[user_id].append(current_time)
    
    window_start = current_time - 60
    user_history[user_id] = [t for t in user_history[user_id] if t > window_start]
    
    tx_count = len(user_history[user_id])
    
    if tx_count > 3:
        print(f"[ALERT] Użytkownik {user_id} wykonał {tx_count} transakcji w ciągu minuty!")
    

Overwriting consumer_anomalies.py
